In [1]:
# Dependencies
import pickle
import numpy as np

import sys
sys.path.append("/home/rguo_hpc/myfolder/mocap")
from datasets.transform import NormalizeConfig, ViewInvariant

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.svm import SVC, LinearSVC

In [2]:
# Load data
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    data_fmr1 = pickle.load(file)

feats = []
for mouse in data_fmr1.keys():
    num_seq = len(data_fmr1[mouse]["ratgen"])
    ratgen  = int(data_fmr1[mouse]["ratgen"][0])
    feat = np.array(data_fmr1[mouse]["m1"]).transpose(0, 1, 3, 2)     # (3, 90000, 3, 23)
    feat = np.delete(feat, [5, 10, 14, 18, 22], axis=-2)
    feats.append(feat)

print(len(feats), feats[0].shape)

8 (3, 90000, 18, 3)


In [3]:
# Reshape to (N, 50, 18, 3)
feats = np.concatenate(feats, axis = 0).reshape(-1, 50, 18, 3)
print(feats.shape)

(43200, 50, 18, 3)


In [ ]:
# view invariant -> augment -> normaliz
vi = ViewInvariant(index_frame = 25, left_idx = 12, right_idx = 15)
feats_norm = np.zeros(feats.shape)

for i in range(len(feats)): 
    feats_norm[i], _, _ =  vi(feats[i], x_supp=(),)

python main.py --mode finetune_last_n --num_prototypes 128 --batch_size 32

# Analysis

In [4]:
L = 4500
tp = 1
l = int(L/tp) # after patch
N = 20
sample_frequency = 9
D = 192
fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]}
fmr1_fold_2 = {"train":[401, 403, 405, 406, 407, 408], "valid": [402, 404]}
fmr1_fold_3 = {"train":[401, 402, 403, 404, 407, 408], "valid": [405, 406]}
fmr1_fold_4 = {"train":[401, 402, 404, 405, 406, 407], "valid": [403, 408]}

In [5]:
# load original data 
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    result = pickle.load(file)

for mouse in result.keys():     # result.keys(): [408, 407, 404, 403, 402, 406, 405, 401]
    num_seq = len(result[mouse]["ratgen"])
    ratgen  = int(result[mouse]["ratgen"][0])
    result[mouse]["ratgen"] = [ratgen for i in range(num_seq * N,)]
    ratid = result[mouse]["ratid"][0]
    result[mouse]["ratid"] = [ratid for i in range(num_seq * N,)]
    
    # llac    
    llac = np.array(result[mouse]["llac"])
    llac = np.squeeze(llac, axis=2)
    llac = llac.reshape(-1, L,)
    if tp > 1:
        grouped_llac = llac.reshape(-1, l, tp) # choose the mode label of each patch 
        result[mouse]["llac"] = mode(grouped_llac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["llac"] = llac
    
    # hlac
    hlac = np.array(result[mouse]["hlac"]) # (3, 90000, 1)
    hlac = np.squeeze(hlac, axis=2)
    hlac = hlac.reshape(-1, L,)
    if tp > 1:
        grouped_hlac = hlac.reshape(-1, l, 3)
        result[mouse]["hlac"] = mode(grouped_hlac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["hlac"] = hlac
    
    del result[mouse]["m1"]

In [6]:
# Train: mouse, genotype, hlac, llac
mouse_tr, gen_tr, hlac_tr, llac_tr = [], [], [], []
for mouse_id in fmr1_fold_1["train"]:
    mouse_tr.append(result[mouse_id]["ratid"])
    gen_tr.append(result[mouse_id]["ratgen"])
    hlac_tr.append(result[mouse_id]["hlac"])
    llac_tr.append(result[mouse_id]["llac"])
    
mouse_tr = np.repeat(np.concatenate(mouse_tr), l)[::sample_frequency]
gen_tr =  np.repeat(np.concatenate(gen_tr), l)[::sample_frequency]
hlac_tr = np.concatenate(hlac_tr)[:,::sample_frequency]
llac_tr = np.concatenate(llac_tr)[:,::sample_frequency]

# Val:  mouse, genotype, hlac, llac
mouse_val, gen_val, hlac_val, llac_val = [], [], [], []
for mouse_id in fmr1_fold_1["valid"]:
    mouse_val.append(result[mouse_id]["ratid"])
    gen_val.append(result[mouse_id]["ratgen"])
    hlac_val.append(result[mouse_id]["hlac"])
    llac_val.append(result[mouse_id]["llac"]) 
    
mouse_val = np.repeat(np.concatenate(mouse_val), l)[::sample_frequency]
gen_val = np.repeat(np.concatenate(gen_val), l)[::sample_frequency]
hlac_val = np.concatenate(hlac_val)[:,::sample_frequency]
llac_val = np.concatenate(llac_val)[:,::sample_frequency]

hlac_tr = hlac_tr.reshape(hlac_tr.size, )
hlac_val = hlac_val.reshape(hlac_val.size, )

llac_tr = llac_tr.reshape(llac_tr.size, )
llac_val = llac_val.reshape(llac_val.size, )

In [7]:
np.load("../swav_output/new_representations_train.npy").shape

(360, 4560, 192)

In [28]:
tr_feats = np.load("../swav_output/new_representations_train.npy")[:, 25:4525][:,::sample_frequency]
val_feats = np.load("../swav_output/new_representations_valid.npy")[:, 25:4525][:,::sample_frequency]
tr_feats = tr_feats.reshape(-1, 192)
val_feats = val_feats.reshape(-1, 192)

In [ ]:
"""
clf = LinearSVC(C=1.0)
clf.fit(tr_feats, hlac_tr)
y_pred = clf.predict(val_feats)
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))
"""

In [29]:
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
model.fit(tr_feats, hlac_tr)        # fit
y_pred = model.predict(val_feats)   # predict

print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.7447666666666667

Classification Report:
               precision    recall  f1-score   support

           1       0.74      0.71      0.72     11792
           2       0.68      0.70      0.69     16949
           3       0.77      0.85      0.81      2183
           4       0.78      0.76      0.77      2158
           5       0.48      0.43      0.45       961
           6       0.92      0.94      0.93      6165
           7       0.72      0.72      0.72     12772
           8       0.79      0.80      0.80      7020

    accuracy                           0.74     60000
   macro avg       0.74      0.74      0.74     60000
weighted avg       0.74      0.74      0.74     60000



/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


- Pretrained 15 epochs ckpt: 0.7310 
- Finetune last 2 blocks, 128_prototypes. Result of finetune 10, 15, 20 epochs  

    epochs    1_blk   `2_blks` 3_blks     4_blks   128_batch   pool  
    10        0.7350   0.7444   0.7382    0.7378   0.7412      0.7372
    15        0.7346   0.7438   0.7350    0.7382   0.7441      ----
    20        ----     0.7421   ----      ----     0.7462      0.7370

    192_prototypes   64_prototypes  last_1_blk() 15_epochs_ckpt   32 batch   gmm init    
    0.7398           0.7391         0.7404       0.7424           0.7385     0.7388
    ----             0.7391         0.7407       0.7388           0.7393     0.7388    
    0.7342           ----           0.7402       ----             ----       ----   

- Pretrained with RoPE, 5 epochs ckpt. 0.7086
- View 2 generated by down sample rate 5
- Loss on pooled representations

    epochs   original    10_eps_ckpt    10_eps_ckpt_128   len_150_10eps_128
                                           fold1/ f2
    10       0.7418      0.7478            0.7478/ 0.7483    0.70
    15       0.7454      0.7455            0.7476
    20       0.7402      0.744             0.7471

with Rope num_frmaes = 60
pretrained model 10 epochs
batch size 128 

    epochs   original       3 views (5, 10)
    10                      73.28    
    15       72.7           73.03